[![Roboflow Notebooks](https://storage.vi.wowza.com/assets/wowza-nb-banner.png)](https://www.wowza.com/)

# License Notice
---

This notebook is derived from examples demonstrating usage of the rf-detr project
(https://github.com/roboflow/rf-detr
), which is licensed under the Apache License 2.0.

The portions of this notebook derived from rf-detr are licensed under the Apache License 2.0.

Modifications and additional content are © 2026 Wowza.

A copy of the Apache License 2.0 is included in licenses/Apache-2.0.txt.

This project is not affiliated with, endorsed by, or sponsored by Roboflow.


# How to Train RF-DETR Object Detection on a Custom Dataset

---

[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WowzaMediaSystems/wse-vi-framework/blob/main/how_to_finetune_rf_detr_on_detection_dataset.ipynb)
[![code](https://badges.aleen42.com/src/github.svg)](https://github.com/WowzaMediaSystems/wowza-video-intelligence-framework)

RF-DETR is a real-time transformer architecture for object detection and instance segmentation developed by Roboflow. Built on a DINOv2 vision transformer backbone, RF-DETR delivers state-of-the-art accuracy and latency trade-offs on [Microsoft COCO](https://cocodataset.org/#home) and [RF100-VL](https://github.com/roboflow/rf100-vl).

RF-DETR uses a DINOv2 vision transformer backbone and supports both detection and instance segmentation in a single, consistent API. All core models and code are released under the Apache 2.0 license.

<img alt="rf_detr_1-4_latency_accuracy_object_detection" src="https://storage.googleapis.com/com-roboflow-marketing/rf-detr/rf_detr_1-4_latency_accuracy_object_detection.png" />

# Important considerations

## Dataset labeling

RF-DETR expects the dataset to be in COCO format. Divide your dataset into three subdirectories: `train`, `valid`, and `test`. Each subdirectory should contain its own `_annotations.coco.json` file that holds the annotations for that particular split, along with the corresponding image files. Below is an example of the directory structure:

```
dataset/
├── train/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
├── valid/
│   ├── _annotations.coco.json
│   ├── image1.jpg
│   ├── image2.jpg
│   └── ... (other image files)
└── test/
    ├── _annotations.coco.json
    ├── image1.jpg
    ├── image2.jpg
    └── ... (other image files)
```

You can achieve that either by leveraging Roboflow's SaaS platform (recommended if ease of use is your main concern) or any other dataset labeling tools that output labels in COCO format (ie https://www.makesense.ai, a browser-local tool).

## Image pre-processing

Another important consideration when preparing a dataset is image pre-processing. RF-DETR models use specific square image resolutions based on the model variant you select (see below), and it's strongly recommended to resize your dataset images to fit the expected dimensions (using letterboxing to maintain aspect ratio). Other transformations can help improve the results of your training process (like auto-orientation and applying small random rotations to your input images).

As mentioned before, Roboflow's platform is the preferred tool if you want to use a simple and intuitive interface that supports everything above, however it's still entirely possible to process your images manually (or via a custom script) and skip Roboflow's platform entirely. The remainder of this notebook will use Roboflow's platform to keep a common baseline highlighting all the necessary steps.

## Environment setup

### Configure API Key

To download the dataset you prepared on Roboflow, you need to provide your Roboflow API key. Follow these steps:

- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy` to copy your private API key.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store your Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [ ]:
import os
from google.colab import userdata

os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")

### Check GPU availability

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `T4 GPU`, and then click `Save`.

In [ ]:
!nvidia-smi

### Install dependencies

Installs RF-DETR version 1.4.1, along with Supervision for benchmarking and Roboflow for pulling datasets and uploading models to the Roboflow platform.

In [ ]:
!pip install -q rfdetr==1.4.1 supervision roboflow torch==2.8.0

## Download Dataset from Roboflow Universe

[Roboflow](https://roboflow.com/annotate) allows you to create object detection datasets from scratch or convert existing datasets from formats like YOLO, and then export them in COCO JSON format for training. You can also explore [Roboflow Universe](https://universe.roboflow.com/) to find pre-labeled datasets for a range of use cases. The cell below simply takes the url you see when loading the dataset you're interest in through your browser.

In [ ]:
from roboflow import download_dataset

dataset = download_dataset("https://app.roboflow.com/[your-org]/[your-dataset]/[your-dataset-version]", "coco")

## Train RF-DETR on custom dataset

### Choose the right `batch_size`

Different GPUs have different amounts of VRAM (video memory), which limits how much data they can handle at once during training. To make training work well on any machine, you can adjust two settings: `batch_size` and `grad_accum_steps`. These control how many samples are processed at a time. The key is to keep their product equal to 16 — that's our recommended total batch size. For example, on powerful GPUs like the A100, set `batch_size=16` and `grad_accum_steps=1`. On smaller GPUs like the T4, use `batch_size=4` and `grad_accum_steps=4`. We use a method called gradient accumulation, which lets the model simulate training with a larger batch size by gradually collecting updates before adjusting the weights.

### Choose the right model variant

RF-DETR supports 4 model variants with different resolutions:

*   **Nano** - 384x384px
*   **Small** - 512x512px
*   **Medium** - 576x576px
*   **Large** - 704x704px

Bigger models take longer to train and run inference on a single frame, however they tend to be more accurate as they have access to higher definition frames.

### Choose the right epoch
Epochs represent the number of "cycles" over the training dataset that we will perform. Choosing the right epoch is sometimes more an art than science. A good starting value for training Medium models is 20. The key is to monitor training metrics as the code below is running, with a special focus on



```
IoU metric: bbox
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=500 ] = 0.831
```

which is printed every time an epoch is completed.

A good rule of thumb is to stop training incrementally if the `IoU=0.50:0.95` metric stops growing for 5 consecutive epochs.

### Saving model weights

By default the output of the training loop is saved in the rf_detr_training directory within your Google Colab workspace. Remember that your Colab runtime can be deleted after a prolonged period of inactivity, so we strongly recommend to run the code cell below that mounts your Google Drive and saves the training output to a permanent folder within your Drive.

### Resume training

After each epoch the training script writes down a "checkpoint", which represents a snapshot in time of the current training progress. Checkpoint files contain all the model weights, and they are the main output used by the VIF framework to run your own custom model. If you already trained a model through this Notebook, you can run additional training loops (Epochs) by uncommenting the code below (`model = RFDETRMedium(custom_checkpoint=output_dir + '/checkpoint.pth')`) and running the corresponding cell again. The training process will resume from the latest checkpoint and continue improving your model's performance, up to a ceiling depending on dataset/model size. As mentioned above, use the `IoU=0.50:0.95` as a reference to determine whether it's worth spending more time on training.

In [ ]:
# OPTIONAL, run only if you want to save weights in your Google Drive for convenience and
# to make sure they are saved in case the connection to Google Colab is lost
from google.colab import drive
import os

drive.mount('/content/drive')
output_dir = "/content/drive/MyDrive/rfdetr_training"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
from rfdetr import RFDETRMedium

try:
    output_dir
except NameError:
    output_dir = "/content/rfdetr_training"

# Valid options are RFDETRNano(), RFDETRSmall(), RFDETRMedium() and RFDETRLarge()
model = RFDETRMedium()

# If the training process stopped unexpectedly, or you want to train additional
# epochs, you can comment the line above and use this variation instead.
# checkpoint.pth always points to the most recent epoch, but
# checkpoint_best_regular.pth is also useful if you want to start from
# the best performing checkpoint for your model yet
# model = RFDETRMedium(custom_checkpoint=output_dir + '/checkpoint.pth')

model.train(
    dataset_dir=dataset.location,
    epochs=20, batch_size=4,
    grad_accum_steps=4,
    output_dir=output_dir
    )

In [ ]:
from PIL import Image

Image.open(output_dir + "/metrics_plot.png")

## Evaluate Fine-tuned RF-DETR Model

The following steps are optional, you can simply grab the ```checkpoint_best_total.pth``` file and upload it to the `models` directory where your VIF service is deployed. Restart the service and it will automatically detect the new model, optimize it for inference via TensorRT compilation and add it to the list of available models.


Before benchmarking the model, we need to load the best saved checkpoint. To ensure it fits on the GPU, we first need to free up GPU memory. This involves deleting any remaining references to previously used objects, triggering Python’s garbage collector, and clearing the CUDA memory cache.

In [ ]:
import gc
import torch
import weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):

    if not torch.cuda.is_available():
        if verbose:
            print("[INFO] CUDA is not available. No GPU cleanup needed.")
        return

    def get_memory_stats():
        allocated = torch.cuda.memory_allocated()
        reserved = torch.cuda.memory_reserved()
        return allocated, reserved

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

    # Ensure we drop all strong references
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print("[WARNING] Object not fully garbage collected yet.")

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    torch.cuda.synchronize()

    if verbose:
        alloc, reserv = get_memory_stats()
        print(f"[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB")

In [ ]:
cleanup_gpu_memory(model, verbose=True)

We load the best-performing model from the `checkpoint_best_total.pth` file using the `RFDETRMedium` class. This checkpoint contains the trained weights from our most successful training run. After loading, we call `optimize_for_inference()`, which prepares the model for efficient inference.

In [ ]:
from rfdetr import RFDETRMedium

model = RFDETRMedium(pretrain_weights=output_dir + "/checkpoint_best_total.pth")
model.optimize_for_inference()

In [ ]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=f"{dataset.location}/test",
    annotations_path=f"{dataset.location}/test/_annotations.coco.json",
)

In [ ]:
import supervision as sv
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision
from PIL import Image

targets = []
predictions = []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)

    targets.append(annotations)
    predictions.append(detections)

In [ ]:
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

## Run Inference with Fine-tuned RF-DETR Model

In [ ]:
import supervision as sv
from PIL import Image

path, image, annotations = ds[0]
image = Image.open(path)

detections = model.predict(image, threshold=0.5)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    "#ffff00", "#ff9b00", "#ff66ff", "#3399ff", "#ff66b2", "#ff8080",
    "#b266ff", "#9999ff", "#66ffff", "#33ff99", "#66ff66", "#99ff00"
])

bbox_annotator = sv.BoxAnnotator(color=color,thickness=thickness)
label_annotator = sv.LabelAnnotator(
    color=color,
    text_color=sv.Color.BLACK,
    text_scale=text_scale)

annotations_labels = [
    f"{ds.classes[class_id]}"
    for class_id
    in annotations.class_id
]

detections_labels = [
    f"{ds.classes[class_id]} {confidence:.2f}"
    for class_id, confidence
    in zip(detections.class_id, detections.confidence)
]

annotation_image = image.copy()
annotation_image = bbox_annotator.annotate(annotation_image, annotations)
annotation_image = label_annotator.annotate(annotation_image, annotations, annotations_labels)

detections_image = image.copy()
detections_image = bbox_annotator.annotate(detections_image, detections)
detections_image = label_annotator.annotate(detections_image, detections, detections_labels)

sv.plot_images_grid(images=[annotation_image, detections_image], grid_size=(1, 2), titles=["Annotation", "Detection"])

<div align="center">
  <p>
    Looking for more tutorials or have questions?
    Visit the <a href="https://www.wowza.com/docs/wowza-video-intelligence-framework">Wowza Video Intelligence Framework documentation</a>.
  </p>
</div>